<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/student_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest Classification: Will the Student Pass?

This notebook matches the classification example from the slides.

We use student data such as study hours, sleep hours, class attendance, and previous grade to predict whether a student will pass or fail.

The goal is to show:

- how a decision tree makes rules
- how a random forest combines many trees
- how bootstrapping creates different training samples
- how feature randomness makes the trees different
- how majority voting gives the final classification

## 1. Import libraries

We use `pandas` for tables, `matplotlib` for plots, and `scikit-learn` for the decision tree and random forest models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

## 2. Create the student dataset

Each row is one student.

The target column is `passed`:

- `1` means the student passed
- `0` means the student failed

This is a classification problem because the output is a category.

In [ ]:
students = pd.DataFrame({
    "study_hours":     [1, 2, 1, 3, 2, 4, 5, 6, 7, 8, 3, 4, 6, 7, 8, 5, 2, 9, 1, 6],
    "sleep_hours":     [5, 5, 7, 4, 6, 6, 7, 8, 7, 8, 5, 7, 6, 8, 9, 5, 8, 7, 4, 9],
    "attended_class":  [0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1],
    "previous_grade": [45, 50, 48, 52, 55, 60, 65, 70, 78, 85, 58, 62, 68, 75, 88, 59, 54, 90, 40, 72],
})

# Simple rule to create labels for the lesson.
# In a real project, these labels would come from historical exam results.
students["passed"] = (
    (students["study_hours"] >= 4) &
    (students["sleep_hours"] >= 6) &
    (students["attended_class"] == 1)
).astype(int)

students

## 3. Visualize the data

Here we plot study hours and sleep hours.

Blue points are students who passed.
Orange points are students who failed.

This helps us see that the data is not best explained by one simple rule. A tree can create several smaller rules instead.

In [ ]:
plt.figure(figsize=(8, 5))
for label, group in students.groupby("passed"):
    name = "Pass" if label == 1 else "Fail"
    plt.scatter(group["study_hours"], group["sleep_hours"], label=name, s=90)

plt.xlabel("Study hours")
plt.ylabel("Sleep hours")
plt.title("Student data: pass or fail")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Split features and target

`X` contains the input features.

`y` contains the output we want to predict.

In [ ]:
features = ["study_hours", "sleep_hours", "attended_class", "previous_grade"]
X = students[features]
y = students["passed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 5. Train one decision tree

A single decision tree creates rules by splitting the data.

For example, it may ask questions like:

- Did the student study more than a certain number of hours?
- Did the student sleep more than a certain number of hours?
- Did the student attend class?

The final prediction is found at a leaf node.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

pred_tree = tree.predict(X_test)
print("Decision tree accuracy:", accuracy_score(y_test, pred_tree))

## 6. Plot the decision tree

This graphic shows the rules learned by one tree.

Each internal node is a question.
Each branch is an answer.
Each leaf is a final prediction.

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree,
    feature_names=features,
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("One Decision Tree")
plt.show()

## 7. Train a random forest

A random forest trains many decision trees.

The trees are intentionally made different in two ways:

1. Each tree gets a different bootstrap sample of the rows.
2. At each split, each tree only sees a random subset of the features.

This creates trees that make different mistakes.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    random_state=42,
    bootstrap=True
)

forest.fit(X_train, y_train)

pred_forest = forest.predict(X_test)
print("Random forest accuracy:", accuracy_score(y_test, pred_forest))
print()
print(classification_report(y_test, pred_forest, target_names=["Fail", "Pass"]))

## 8. Confusion matrix

The confusion matrix shows where the model was correct and where it was wrong.

In [ ]:
cm = confusion_matrix(y_test, pred_forest)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

## 9. Majority voting

For classification, every tree votes.

If most trees vote `Pass`, the forest predicts `Pass`.
If most trees vote `Fail`, the forest predicts `Fail`.

Here we look at one test student and count the votes from all trees.

In [ ]:
student_index = X_test.index[0]
one_student = X_test.loc[[student_index]]

print("Student features:")
display(one_student)

votes = np.array([estimator.predict(one_student)[0] for estimator in forest.estimators_])
fail_votes = np.sum(votes == 0)
pass_votes = np.sum(votes == 1)

print("Fail votes:", fail_votes)
print("Pass votes:", pass_votes)
print("Final prediction:", "Pass" if forest.predict(one_student)[0] == 1 else "Fail")

## 10. Plot the votes

This plot shows the majority vote for one student.

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(["Fail", "Pass"], [fail_votes, pass_votes])
plt.ylabel("Number of trees")
plt.title("Random Forest Majority Vote")
plt.show()

## 11. Feature importance

Random forests can estimate which features were most useful for the predictions.

A higher value means the feature helped the trees reduce impurity more often.

In [ ]:
importance = pd.Series(forest.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance.index, importance.values)
plt.xlabel("Feature importance")
plt.title("Which features mattered most?")
plt.show()

importance.sort_values(ascending=False)

## 12. Decision boundary using only two features

To draw a 2D decision boundary, we train a smaller random forest using only:

- study hours
- sleep hours

The model divides the space into regions where it predicts `Pass` or `Fail`.

In [ ]:
X_two = students[["study_hours", "sleep_hours"]]
y_two = students["passed"]

forest_two = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
forest_two.fit(X_two, y_two)

x_min, x_max = X_two["study_hours"].min() - 1, X_two["study_hours"].max() + 1
y_min, y_max = X_two["sleep_hours"].min() - 1, X_two["sleep_hours"].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = pd.DataFrame({
    "study_hours": xx.ravel(),
    "sleep_hours": yy.ravel()
})
Z = forest_two.predict(grid).reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z, alpha=0.25)

for label, group in students.groupby("passed"):
    name = "Pass" if label == 1 else "Fail"
    plt.scatter(group["study_hours"], group["sleep_hours"], label=name, s=90, edgecolor="black")

plt.xlabel("Study hours")
plt.ylabel("Sleep hours")
plt.title("Random Forest Decision Boundary")
plt.legend()
plt.show()

## 13. Bootstrapping example

Bootstrapping means we sample rows with replacement.

If the original dataset has six rows:

`A, B, C, D, E, F`

One bootstrap sample might be:

`C, A, F, C, E, A`

Notice that:

- the order changed
- some rows repeat
- some rows are missing

But the bootstrap sample still has the same number of rows as the original dataset.

In [ ]:
original_rows = np.array(["A", "B", "C", "D", "E", "F"])
np.random.seed(7)
bootstrap_sample = np.random.choice(original_rows, size=len(original_rows), replace=True)

print("Original rows:        ", list(original_rows))
print("Bootstrap sample:    ", list(bootstrap_sample))

## 14. Summary

A random forest is powerful because it combines many different decision trees.

For classification:

- each tree predicts a category
- the forest uses majority voting

The trees become different because of:

- bootstrapping: different random samples of rows
- feature randomness: different subsets of features at each split

Because the trees make different mistakes, the final forest is usually more stable than one decision tree.